# Critical Care Survival starter

**Will this patient be alive in two months?** Seriously ill adults of the
SUPPORT cohort, described the way five hospital charts recorded them on the
third day.

In this challenge **the model is fixed**. The scorer always fits
scikit-learn's default `LogisticRegression()` on the features you submit,
and ranks the test patients by ROC AUC. You do not submit predictions; you
submit a **feature matrix**. Every point of AUC comes from preprocessing.

**The cohort.** SUPPORT (Study to Understand Prognoses and Preferences for
Outcomes and Risks of Treatments) followed 9,105 seriously ill adults
admitted to five US academic medical centres between 1989 and 1994. The
physiology you receive is the worst value recorded on the third day after
study entry, in 31 columns as a hospital export would look. The label is the
patient's vital status 60 days after entry: `dead = 1` if the patient died
within that window. The outcome is modelled from the clinical picture, so it
is not in the public file and cannot be looked up.

**The ladder.** Every row is the same `LogisticRegression()` on the official
split. Only the features change.

| features | test AUC |
|---|---|
| a constant column ("always alive") | 0.500 |
| the numeric columns as pandas reads them, empty cells set to 0, unscaled | 0.757 (does not converge) |
| **the benchmark:** the same columns, median-imputed and standardised | **0.850** (train AUC 0.962) |
| + repaired values, units harmonised per site, the post-outcome column dropped | 0.877 |
| + labs imputed as the investigators did, absences kept as information | 0.900 |
| + categories encoded as categories, orders as orders | 0.916 |
| + the bedside formulas | 0.937 |
| *gradient boosting on the raw columns, for reference* | *0.907* |

**The pass bar for this module is a test AUC of 0.905.** Each rung is one
kind of knowledge put into numbers. A straight line on good features beats a
flexible model on raw columns.

**How this notebook works.** Sections 1 to 6 are complete: download, read,
the clinical brief, the data analysis, a local evaluation helper, a
submission helper. Steps 0 and 1 are complete too, and step 1 reaches the
benchmark. Steps 2 to 6 are yours: each has a skeleton with `# TODO` lines.
The skeletons run as they are and return the step-1 matrix, so the notebook
executes end to end at any time. Fill them one at a time and watch your own
ladder grow.

Read `EXPERTISE.md` before you decide anything. Section 3 summarises it.

---

## 0. Setup

The ML-Arena client is published as **`mlarena-sdk`** and imports as
`mlarena`. Do not `pip install mlarena`: that is an unrelated package.

The scorer runs **scikit-learn 1.8.0**. Install the same version so that
your local numbers match the leaderboard.

In [ ]:
!pip install -q mlarena-sdk scikit-learn==1.8.0

---

## 1. Get the data

Paste your personal API key from your ML-Arena **Profile** page (it starts
with `mlk_user_`) and the challenge id, the number at the end of the
challenge page's address. `download_dataset` writes five files into the
working directory: `train.csv.gz`, `test.csv.gz`, `sample_submission.csv.gz`,
`EXPERTISE.md` and `DICTIONARY.md`.

In [ ]:
import mlarena

API_KEY = "mlk_user_..."   # <- paste yours here
CHALLENGE_ID = 192        # <- the number at the end of the challenge page URL

assert CHALLENGE_ID is not None, "set CHALLENGE_ID to the number in the challenge page URL"
client = mlarena.connect(api_key=API_KEY)
client.download_dataset(CHALLENGE_ID, ".")

---

## 2. Read it

`train.csv.gz` has the target `dead` (1 = died within 60 days);
`test.csv.gz` has the same columns without it. pandas reads the gzip
directly.

In [ ]:
import numpy as np
import pandas as pd
from pandas.api.types import is_numeric_dtype

train = pd.read_csv("train.csv.gz", low_memory=False)
test = pd.read_csv("test.csv.gz", low_memory=False)
y = train["dead"]

print("train", train.shape, " test", test.shape)
print(f"death rate in train: {y.mean():.3f}")
train.head()

One row per patient, 31 columns. Look at what pandas made of each column
before deciding anything. `is_numeric_dtype` is the reliable test: with
pandas 3, text columns are `str`, not `object`.

The table below is sorted by fill rate. **Read the `dtype` column
carefully.** One column that should hold a number was read as text. Which
one is it, and why did pandas do that? Find the answer in the data before
you open `DICTIONARY.md`: look at the column's distinct values. Then ask
what happens to that column in a feature matrix built from "the numeric
columns", and what `fillna(0)` would have done to it.

In [ ]:
features = [c for c in train.columns if c not in ("id", "dead")]
numeric = [c for c in features if is_numeric_dtype(train[c])]
text = [c for c in features if c not in numeric]
print(len(numeric), "numeric columns,", len(text), "text columns:", text)

summary = pd.DataFrame({
    "dtype": train[features].dtypes.astype(str),
    "filled_train": train[features].notna().mean().round(3),
    "filled_test": test[features].notna().mean().round(3),
    "distinct": train[features].nunique(),
})
summary.sort_values("filled_train")

---

## 3. What clinicians know

`EXPERTISE.md` is twelve numbered facts. This section condenses the ones
you will use, with the numbers. It does not say which fact is worth how
much: the ladder gives what the steps are worth together, and measuring each
one is your job.

### 3.1 Reference ranges (adults)

| column | normal | unit | a low value means | a high value means |
|---|---|---|---|---|
| `meanbp` | 70 to 100 | mmHg | shock, circulatory failure (0 = arrest) | hypertensive crisis |
| `hrt` | 60 to 100 | /min | bradycardia, arrest at 0 | tachycardia: sepsis, shock, arrhythmia (300 is about the ceiling) |
| `resp` | 12 to 20 | /min | respiratory depression | respiratory distress |
| `temp` | 36.5 to 37.5 | °C | hypothermia (31.7 is severe but real) | fever, infection |
| `wblc` | 4 to 11 | ×10⁹/L | marrow failure, overwhelming sepsis | infection, leukaemia (up to 200) |
| `pafi` | above 400 on room air; above 300 is not ARDS | mmHg | oxygenation failure (ARDS, pneumonia) | |
| `alb` | 3.5 to 5.0 | g/dL | malnutrition, liver failure, chronic inflammation | above about 6 does not occur |
| `bili` | 0.1 to 1.2 | mg/dL | | liver failure, biliary obstruction (60 is fulminant) |
| `crea` | 0.6 to 1.2 | mg/dL (53 to 106 µmol/L) | | kidney failure (20 mg/dL is dialysis level) |
| `bun` | 7 to 20 | mg/dL | | kidney failure, or a kidney that is under-perfused |
| `sod` | 135 to 145 | mEq/L | hyponatraemia (fluid overload) | hypernatraemia (dehydration) |
| `ph` | 7.35 to 7.45 | | acidaemia (shock, kidney or respiratory failure) | alkalaemia |
| `glucose` | 70 to 100 fasting | mg/dL | hypoglycaemia | hyperglycaemia (diabetes, stress; 1000 is a hyperosmolar state) |
| `urine` | 800 to 2000 | mL/day | oliguria below 500, anuria below 100 (kidney injury, shock) | polyuria |
| `scoma` | 0 | 0 to 100 | | deeper coma (100 = deep coma) |

*EXPERTISE.md facts 4 and 7; Harrison's appendix; MSD Manual.*

### 3.2 A lab that was not ordered is presumed normal (fact 2)

A test is ordered when the physician wants its result. When none was
ordered, the SUPPORT investigators did not treat the value as unknown: they
filled it with a normal value, and they published the values they used.

| lab | SUPPORT fill-in | unit |
|---|---|---|
| `alb` | 3.5 | g/dL |
| `pafi` | 333.3 | mmHg |
| `bili` | 1.01 | mg/dL |
| `crea` | 1.01 | mg/dL |
| `bun` | 6.51 | mg/dL |
| `wblc` | 9 | ×10⁹/L |
| `urine` | 2502 | mL/day |

Arterial pH and glucose are not in their list; the textbook normals are pH
7.40 and glucose about 100 mg/dL.

**Question: is the cohort median a normal value?** Filling an empty lab with
the median asserts that the untested patient looks like the tested ones.
The cell below puts the medians next to the fill-ins. For which labs do the
two differ most, and in which direction?

### 3.3 Which columns travel together (fact 3)

Labs are ordered as panels, so they are missing as panels:

- **the arterial blood gas**: `pafi` and `ph`, one sample, drawn when there is
  a respiratory or acid-base concern;
- **the liver panel**: `alb` and `bili`;
- **the metabolic work-up**: `glucose`, `bun` and the 24-hour `urine`
  collection;
- **the social interview**: `edu` and `income`.

Two columns report the same thing from two people. `adlp` is the number of
daily activities the *patient* said they could not do; it is empty when the
patient could not be interviewed on day 3 (intubated, comatose, delirious).
`adls` is the same count from a *surrogate*, usually family. The SUPPORT
team built one ADL score: the patient's answer when it exists, the
surrogate's otherwise.

For each column, decide what empty means: "unknown", "not ordered because
nothing suggested it", or "the patient could not speak", and whether the
fact of the absence deserves a column of its own.

### 3.4 Sites chart in different units (fact 5)

- **Creatinine**: sites A, B and C report mg/dL; **sites D and E report
  µmol/L**. 1 mg/dL = 88.4 µmol/L.
- **Temperature**: sites A, B, D and E chart °C; **site C charts °F**.
  °F = °C × 9/5 + 32.

Any threshold, ratio or comparison across sites is meaningless until the
units agree. Harmonise first, then look for extremes: a creatinine of 700
is impossible in mg/dL and ordinary in µmol/L.

### 3.5 What clinicians compute at the bedside (fact 7)

A clinician computes these from the chart. A straight line cannot compute a
ratio, a threshold or a logarithm from raw columns.

- **SIRS count** (ACCP/SCCM 1992), 0 to 4: one point each for temperature
  above 38 or below 36 °C, heart rate above 90 /min, respiratory rate above
  20 /min, white cells above 12 or below 4 ×10⁹/L.
- **Shock index** = heart rate / systolic pressure (normal 0.5 to 0.7).
  Systolic pressure is not in the file; the **modified shock index** =
  heart rate / mean arterial pressure is (normal about 0.7 to 1.3, mortality
  rises above 1.3).
- **BUN/creatinine ratio**, both in mg/dL: above 20 suggests prerenal
  azotaemia (the kidney is under-perfused rather than damaged).
- **Berlin classes of ARDS** from PaO2/FiO2: above 300 none, 200 to 300
  mild, 100 to 200 moderate, below 100 severe.
- **Oliguria**: fewer than 500 mL of urine in 24 hours; anuria fewer than
  100.
- **Hypoalbuminaemia**: albumin below 3.5 g/dL (below 2.5 severe).
  **Acidaemia**: pH below 7.35; alkalaemia above 7.45. **Dysnatraemia**:
  sodium outside 135 to 145, in either direction.
- **Log scale** for bilirubin, creatinine, BUN and white cells: they span
  two orders of magnitude and clinicians think in doublings, not
  differences. A white cell count is abnormal in both directions.
- `hday`, the hospital day of entry, is also skewed: most patients entered
  on day 1, a few after weeks (fact 11).

### 3.6 Impossible values versus real extremes (fact 4)

Seriously ill patients have extreme values, and most extremes in this file
are real: a mean pressure of 0 (arrest), a heart rate of 300, a white cell
count of 200 (leukaemia), a bilirubin of 60, a creatinine of 20 mg/dL, a
glucose of 1000, a urine output of 0 (anuria) or 9000. A few values are not
measurements at all: an age of 999 (a placeholder for "unknown"), a
temperature of 0.0 (not taken), a sodium of 1370 (a decimal slip for 137.0).
A clipping rule written without the physiology removes real patients; a
placeholder left in place is a lever on a straight line.

### 3.7 Billed at discharge (fact 10)

`charges` is the total hospital bill for the stay. It is computed at
discharge, grows with the length and intensity of the stay, and so also
reflects how the stay ended. For a patient still in the ward there is no
bill yet. The training cohort is historical and fully billed; the test
patients are still admitted at extraction. A model cannot use at prediction
time what does not exist at prediction time.

### 3.8 Categories and orders (fact 6)

`dzgroup` is the qualifying diagnosis in eight groups; a coma and a lung
cancer kill through different mechanisms and at different speeds, so they
are not points on one scale. `dzclass` groups the same column into four and
contains nothing that `dzgroup` does not. Two columns *are* orders: `ca`
(no < yes < metastatic) and `income` (four brackets). `sex` and `race` are
nominal; the SUPPORT prognostic model did not use them.

In [ ]:
SUPPORT_NORMAL = {"alb": 3.5, "pafi": 333.3, "bili": 1.01, "crea": 1.01, "bun": 6.51,
                  "wblc": 9.0, "urine": 2502.0, "ph": 7.40, "glucose": 100.0}

rows = []
for col, normal in SUPPORT_NORMAL.items():
    s = pd.to_numeric(train[col], errors="coerce")        # a text token, if any, becomes NaN here
    if col == "crea":
        s = s[train["site"].isin(["A", "B", "C"])]        # the mg/dL sites only (section 3.4)
    rows.append({"lab": col, "fill-in (normal)": normal, "cohort median": round(s.median(), 2),
                 "filled": round(s.notna().mean(), 3)})
pd.DataFrame(rows).set_index("lab")

---

## 4. Look at the data

Eight views, each followed by questions. Answer them in a markdown cell of
your own: the answers are the decisions of steps 2 to 6.

### (a) Missingness per column, train and test side by side

In [ ]:
import matplotlib.pyplot as plt

BLUE, ORANGE = "#2a78d6", "#eb6834"

missing = pd.DataFrame({
    "train": train[features].isna().mean(),
    "test": test[features].isna().mean(),
}).sort_values("train")

fig, ax = plt.subplots(figsize=(8, 9))
pos = np.arange(len(missing))
ax.barh(pos - 0.2, missing["train"], height=0.4, color=BLUE, label="train")
ax.barh(pos + 0.2, missing["test"], height=0.4, color=ORANGE, label="test")
ax.set_yticks(pos, list(missing.index))
ax.set_xlabel("share of empty cells")
ax.set_title("Missingness per column")
ax.grid(axis="x", alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

**Questions.** One column is empty in every test row and filled in almost
every train row. What is it, what does it mean clinically (section 3.7),
and what must you do with it? A second column looks 63 % filled here: how
many of those "filled" cells are numbers? (pandas counts a text token as a
value.)

### (b) The same lab at five sites

Creatinine (log scale) and temperature, one box per site.

In [ ]:
sites = sorted(train["site"].unique())
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, col in zip(axes, ["crea", "temp"]):
    data = [train.loc[train["site"] == s, col].dropna() for s in sites]
    ax.boxplot(data, flierprops={"marker": ".", "markersize": 4, "alpha": 0.5})
    ax.set_xticks(range(1, len(sites) + 1), sites)
    ax.set_xlabel("site")
    ax.set_title(f"{col} by site (train)")
    ax.grid(axis="y", alpha=0.3)
axes[0].set_yscale("log")
axes[0].set_ylabel("crea, as charted (log scale)")
axes[1].set_ylabel("temp, as charted")
plt.tight_layout()
plt.show()

**Questions.** Same lab, five hospitals: why two clusters? Which sites
belong together, and by what factor do the clusters differ? What are the
points at 0 on the temperature plot, and are they at one site or all of
them?

### (c) Three labs on a linear and on a log scale

Bilirubin, creatinine (sites A, B, C, so that one unit is shown) and BUN.

In [ ]:
labs = {
    "bili": train["bili"],
    "crea (sites A-C)": train.loc[train["site"].isin(["A", "B", "C"]), "crea"],
    "bun": train["bun"],
}
fig, axes = plt.subplots(2, 3, figsize=(12, 6))
for j, (name, s) in enumerate(labs.items()):
    s = s.dropna()
    axes[0, j].hist(s, bins=60, color=BLUE)
    axes[0, j].set_title(f"{name}, linear")
    axes[1, j].hist(np.log10(s[s > 0]), bins=60, color=BLUE)
    axes[1, j].set_title(f"log10({name})")
for ax in axes.ravel():
    ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

**Questions.** On which scale does a fixed step mean the same thing at the
low end and at the high end? What does a doubling of bilirubin look like on
each? Where would a straight line fitted on the linear column put the
patient with a bilirubin of 63? (The comb of spikes at the low end of the
log plots is the chart's rounding to 0.1, not biology.)

### (d) How `sex` and `race` are spelled

In [ ]:
for col in ["sex", "race"]:
    counts = train[col].value_counts(dropna=False)
    counts.index = [repr(v) for v in counts.index]      # repr shows case and trailing spaces
    print(f"{col}: {len(counts)} distinct spellings")
    print(counts.to_string(), "\n")

**Questions.** How many categories are there really in each column? Which
spellings are the same category? `race` has an empty cell at four sites and
the string `unknown` at one: are those the same thing? Does the SUPPORT
model use either column (section 3.8)?

### (e) Death rate by disease group and by cancer status

In [ ]:
by_group = train.groupby("dzgroup")["dead"].mean().sort_values()
by_ca = train.groupby("ca")["dead"].mean().reindex(["no", "yes", "metastatic"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={"width_ratios": [2, 1]})
axes[0].barh(list(by_group.index), by_group.values, color=BLUE)
axes[0].set_xlabel("death rate")
axes[0].set_title("by disease group (dzgroup)")
axes[1].bar(list(by_ca.index), by_ca.values, color=BLUE)
axes[1].set_ylabel("death rate")
axes[1].set_title("by cancer status (ca)")
for ax in axes:
    ax.grid(axis="x" if ax is axes[0] else "y", alpha=0.3)
plt.tight_layout()
plt.show()

pd.crosstab(train["dzgroup"], train["ca"])

**Questions.** Is disease group an order? Could you rank the eight groups on
one axis and hand the line one number per patient, or does each group need
its own effect? Cancer status is an order by definition (no < yes <
metastatic), yet the raw death rate is not monotonic: `yes` is above
`metastatic`. Does the raw rate settle the question? Look at the crosstab:
which disease groups are the metastatic patients in, and what else differs
between those groups and the rest?

### (f) Extremes: real or impossible?

Age, sodium and temperature: min, max, and the five most extreme values on
each side.

In [ ]:
for col in ["age", "sod", "temp"]:
    s = train[col]
    print(f"{col:5s} min {s.min():>6}  max {s.max():>7}   "
          f"lowest {s.nsmallest(5).round(1).tolist()}   highest {s.nlargest(5).round(1).tolist()}")
print()
print("temp above 45, by site:  ", train.loc[train["temp"] > 45, "site"].value_counts().to_dict())
print("temp equal to 0.0, by site:", train.loc[train["temp"] == 0, "site"].value_counts().to_dict())

**Questions.** Which of these values are patients and which are
bookkeeping? For temperature, check the site before you decide: 105.8 is a
real fever in one unit and not in the other. What would a rule like "clip
every column to its 1st and 99th percentile" do to the heart rate of 300 and
the mean pressure of 0 (section 3.6)? What does a single 999 do to a
straight line fitted on `age`?

### (g) Is an absence informative?

Death rate when a value is present versus missing, for four kinds of
absence.

In [ ]:
absences = {
    "patient ADL\n(adlp)": train["adlp"].isna(),
    "blood gas\n(pafi and ph)": train["pafi"].isna() & train["ph"].isna(),
    "liver panel\n(alb)": train["alb"].isna(),
    "metabolic\n(bun)": train["bun"].isna(),
    "interview\n(income)": train["income"].isna(),
}
rate_present = [train.loc[~m, "dead"].mean() for m in absences.values()]
rate_missing = [train.loc[m, "dead"].mean() for m in absences.values()]

fig, ax = plt.subplots(figsize=(9, 4))
pos = np.arange(len(absences))
ax.bar(pos - 0.2, rate_present, width=0.4, color=BLUE, label="value present")
ax.bar(pos + 0.2, rate_missing, width=0.4, color=ORANGE, label="value missing")
ax.set_xticks(pos, list(absences))
ax.set_ylabel("death rate")
ax.set_title("Death rate when a value is present versus missing (train)")
ax.grid(axis="y", alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()
print("share missing:", {k.split(chr(10))[0]: round(float(m.mean()), 3) for k, m in absences.items()})

**Questions.** Is every absence informative in the same way? For each pair,
say what the absence means at the bedside (section 3.3) and whether the
difference in death rate is the absence itself or the kind of patient who
gets that test ordered. After you fill an empty lab with a number, can a
straight line still see that it was empty?

### (h) Which columns are missing together

Correlation between the "is missing" indicators of the labs, the two ADL
columns and the interview.

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

lab_cols = ["wblc", "pafi", "ph", "alb", "bili", "glucose", "bun", "urine", "adlp", "adls", "edu", "income"]
M = train[lab_cols].copy()
M["glucose"] = pd.to_numeric(M["glucose"], errors="coerce")   # the text token counts as missing here
M = M.isna().astype(float)
C = M.corr()

diverging = LinearSegmentedColormap.from_list("blue_gray_red", ["#1c5cab", "#f0efec", "#d03b3b"])
fig, ax = plt.subplots(figsize=(7.5, 6.5))
im = ax.imshow(C.values, vmin=-1, vmax=1, cmap=diverging)
ax.set_xticks(range(len(lab_cols)), lab_cols, rotation=90)
ax.set_yticks(range(len(lab_cols)), lab_cols)
for i in range(len(lab_cols)):
    for j in range(len(lab_cols)):
        if i != j and abs(C.values[i, j]) >= 0.3:
            ax.text(j, i, f"{C.values[i, j]:.1f}", ha="center", va="center", fontsize=7)
plt.colorbar(im, ax=ax, label="correlation of 'is missing'")
ax.set_title("Which columns are missing together?")
plt.tight_layout()
plt.show()

**Questions.** Which columns form a block? Match each block to a panel of
section 3.3. If you add an indicator for an absence, is one column per lab
or one per panel the right unit? Which panel is `edu` in?

---

## 5. The evaluation helper

The scorer fits `LogisticRegression()` with its defaults on your train rows
and computes the ROC AUC on the test rows. You do not have the test labels,
so `cv_auc` does the same thing inside train: three stratified folds, the
scorer's exact model, the mean and spread of the AUC on the held-out fold,
the AUC on the fitting fold (the leaderboard's "Train AUC"), and whether
the solver converged. Every call appends a row to `results`, your own
ladder.

The official test rows are a random part of the same cohort, so a random
3-fold split of the train rows estimates the leaderboard number to about
±0.01, on one condition: the held-out rows must look like the test rows in
every column. Step 1 shows what happens when they do not.

In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

results = []   # your own ladder: one row per cv_auc call


def cv_auc(X, y, name):
    """3-fold stratified CV of the scorer's exact model. X: a DataFrame indexed like train."""
    y = y.loc[X.index]
    values = X.to_numpy(dtype=float)
    assert np.isfinite(values).all(), "NaN or inf in X: impute before you evaluate"
    folds = StratifiedKFold(n_splits=3, shuffle=True, random_state=0)
    val_auc, fit_auc, converged = [], [], True
    for fit_idx, val_idx in folds.split(values, y):
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            model = LogisticRegression().fit(values[fit_idx], y.iloc[fit_idx])   # the scorer's model, untouched
        if any(issubclass(w.category, ConvergenceWarning) for w in caught):
            converged = False
        val_auc.append(roc_auc_score(y.iloc[val_idx], model.predict_proba(values[val_idx])[:, 1]))
        fit_auc.append(roc_auc_score(y.iloc[fit_idx], model.predict_proba(values[fit_idx])[:, 1]))
    row = {"step": name, "cv_auc": round(float(np.mean(val_auc)), 4), "std": round(float(np.std(val_auc)), 4),
           "train_auc": round(float(np.mean(fit_auc)), 4), "features": values.shape[1], "converged": converged}
    results.append(row)
    print(f"{name}\n  CV AUC {row['cv_auc']:.4f} ± {row['std']:.4f}   train AUC {row['train_auc']:.4f}   "
          f"features {row['features']}   converged {converged}")
    return row["cv_auc"]

---

## 6. The submission helper

One file, **`submission.csv.gz`**, gzip-compressed:

- column `id`, then 1 to 300 numeric feature columns with no NaN or inf;
- **every** test id;
- train ids: all of them, or any subset of **at least 4,000** (dropping
  rows you do not trust is allowed);
- the scorer uses its own copy of the labels, so do not include `dead`.

`write_submission` takes the two matrices (DataFrames indexed like `train`
and `test`, same columns in the same order), checks each rule the scorer
enforces, and writes the file. A rejected upload costs one of your daily
submissions, so the checks run here first. `to_csv` compresses from the
`.gz` extension; `index=False` matters, an unnamed index column is
rejected.

In [ ]:
def write_submission(X_train, X_test, path="submission.csv.gz"):
    """Check X_train / X_test against the scorer's rules and write the file."""
    assert isinstance(X_train, pd.DataFrame) and isinstance(X_test, pd.DataFrame), "pass DataFrames"
    assert list(X_train.columns) == list(X_test.columns), "train and test need the same columns in the same order"
    frame = pd.concat([
        X_train.assign(id=train.loc[X_train.index, "id"].to_numpy()),
        X_test.assign(id=test.loc[X_test.index, "id"].to_numpy()),
    ], ignore_index=True)
    frame = frame[["id"] + list(X_train.columns)]
    values = frame.drop(columns="id")
    assert frame["id"].is_unique, "duplicate ids"
    assert set(test["id"]) <= set(frame["id"]), "a test id is missing"
    assert frame["id"].str.startswith("tr_").sum() >= 4_000, "fewer than 4,000 train rows"
    assert 1 <= values.shape[1] <= 300, "1 to 300 feature columns"
    assert values.columns.is_unique, "a column name appears twice"
    assert "dead" not in values.columns, "the target is not a feature"
    assert all(is_numeric_dtype(values[c]) for c in values), "a text column"
    assert np.isfinite(values.to_numpy(dtype=float)).all(), "NaN or inf"
    frame.to_csv(path, index=False)
    print(f"wrote {path}: {frame.shape[0]} rows, {values.shape[1]} feature column(s)")

---

## 7. Step 0: the "always alive" submission

One constant column. The model can only learn the intercept, every patient
gets the same score, and the AUC is 0.5 whatever the labels. This is the
floor of the ladder; the point is to run the plumbing once, end to end,
before anything can go wrong for a better reason.

In [ ]:
X0_train = pd.DataFrame({"always_alive": 0.0}, index=train.index)
X0_test = pd.DataFrame({"always_alive": 0.0}, index=test.index)

cv_auc(X0_train, y, "step 0: a constant column")
write_submission(X0_train, X0_test)

### Submit

The file you upload must be named exactly `submission.csv.gz`. `submit`
returns as soon as the file is deployed; the score appears on the
leaderboard a minute or two later, with the train AUC, the number of
features, the number of train rows and the convergence flag.

**Every later step rewrites `submission.csv.gz`. Come back and run these two
cells whenever you want the official number for the matrix you just built.**

In [ ]:
result = client.submit(challenge_id=CHALLENGE_ID, files=["submission.csv.gz"])
print(result)

In [ ]:
client.leaderboard(CHALLENGE_ID).head(10)

---

## 8. Step 1: baseline++

The numeric columns as pandas reads them, median-imputed and standardised.
This is the benchmark of the ladder, and the last step that knows nothing
about patients.

Two rules that every later step keeps. **One function applied to train and
test alike.** **Everything data-dependent (a median, a scale, a vocabulary)
is learned from the train rows only.** `SimpleImputer` and `StandardScaler`
follow the pattern: `fit` on train, `transform` both. `impute_and_scale`
wraps the two and returns DataFrames; the later steps reuse it.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler


def numeric_columns(frame):
    return [c for c in frame.columns if c not in ("id", "dead") and is_numeric_dtype(frame[c])]


def impute_and_scale(F_train, F_test):
    """Median from the train rows, then z-score from the train rows, applied to both."""
    imputer = SimpleImputer(strategy="median").fit(F_train)
    scaler = StandardScaler().fit(imputer.transform(F_train))

    def transform(F):
        return pd.DataFrame(scaler.transform(imputer.transform(F)), columns=F_train.columns, index=F.index)

    return transform(F_train), transform(F_test)


def build_features_1(train, test):
    cols = numeric_columns(train)
    return impute_and_scale(train[cols], test[cols])


X1_train, X1_test = build_features_1(train, test)
cv_auc(X1_train, y, "step 1: numeric as read, median-imputed, standardised")
write_submission(X1_train, X1_test)

The local check says about **0.96**. The ladder says this exact matrix, the
benchmark, scores **0.850** on the test rows with a train AUC of 0.962.
Submit it and read the leaderboard line for yourself: `WARNING: train AUC is
0.112 above test AUC`.

Two questions before you touch anything.

1. Why did the local check miss by 0.11? A held-out fold stands in for the
   test rows only if it looks like them in every column. Go back to figure
   (a) of section 4.
2. Which column carries the outcome on the train rows and does not exist
   for the test patients? What is it, clinically (section 3.7)? What must
   step 2 do with it?

Until that column is gone, your local number is not an estimate of
anything. Once it is gone, `cv_auc` and the leaderboard agree to about
0.01, and the train AUC sits next to the test AUC instead of far above it.

---

## 9. Steps 2 to 6: your work

Each step below has a recap of what you know (with the fact numbers of
`EXPERTISE.md`), a skeleton with `# TODO` lines, a `cv_auc` call, a
`write_submission` call, and the leaderboard number to aim for. The
skeletons run as they are and return the step-1 matrix; nothing improves
until you fill them. Fill one step, run its cell, compare the CV number
with the target, and run the Submit cells of section 7 when you want the
official number. Keep every decision in the code with a one-line comment
saying why: your notebook is part of the grade.

### Step 2: repair

**What you know.** Fact 4: an age of 999, a temperature of 0.0 and a sodium
of 1370 are not measurements; a heart rate of 300, a mean pressure of 0 and
a white cell count of 200 are. Fact 5: creatinine is in µmol/L at two sites
and temperature in °F at one; 1 mg/dL = 88.4 µmol/L and °C = (°F − 32) ×
5/9. Fact 10: the bill exists only after discharge, and the test patients
have none. `DICTIONARY.md`: `sex` has eight spellings, `race` eleven, and
one numeric column was read as text because one site writes a token in it.

**Aim for 0.877** on the leaderboard. Note where the train AUC lands.

In [ ]:
def repair(df):
    """Return a repaired copy of df. Every rule applies to train and test alike; none needs the target."""
    r = df.copy()
    # TODO the column pandas read as text: make it numeric. pd.to_numeric(r[col], errors="coerce") turns the token into NaN.
    # TODO age: the placeholder for "unknown" is not an age. Set it to NaN (the median fills it later).
    # TODO temp: 0.0 means "not taken", not a temperature. Then convert the site that charts in °F to °C.
    # TODO crea: convert the sites that report µmol/L to mg/dL.
    # TODO sod: repair the decimal slip (a sodium above 500 is the value x 10).
    # TODO sex, race: one spelling per category (.str.strip().str.lower()). What does 'unknown' mean for race?
    # TODO charges: billed at discharge, empty for every test patient. Drop the column.
    return r


def build_features_2(train, test):
    R_train, R_test = repair(train), repair(test)
    cols = numeric_columns(R_train)
    return impute_and_scale(R_train[cols], R_test[cols])


X2_train, X2_test = build_features_2(train, test)
cv_auc(X2_train, y, "step 2: repaired values, units, text, post-outcome column")
write_submission(X2_train, X2_test)

A self-check. Before the repair this prints two clusters per lab, a maximum
age of 999, a sodium above 1000 and a text column. After it: one cluster,
an age below 110, a sodium below 200, and no text column except the
categories.

In [ ]:
R = repair(train)
print(R.groupby("site")[["crea", "temp"]].median().round(2))
print("age max", R["age"].max(), "  sod max", R["sod"].max(), "  temp min", R["temp"].min(),
      "  crea max", R["crea"].max())
print("text columns:", [c for c in R.columns if c != "id" and not is_numeric_dtype(R[c])])
print("sex spellings:", sorted(R["sex"].dropna().unique()))

### Step 3: missing values, with knowledge

**What you know.** Fact 2: a lab that was not ordered was presumed normal
by the investigators, and section 3.2 lists their fill-in values next to
the cohort medians. Fact 3: some absences carry information (the patient
who could not answer, the blood gas nobody needed), and the columns of a
panel are missing together, so one indicator may stand for several columns.
Fact 3 also says how the SUPPORT team combined the two ADL reporters.
Figures (g) and (h) are the evidence. Order matters inside the function: an
indicator must be computed before the fill erases the absence.

**Aim for 0.900.**

In [ ]:
NORMAL = {
    # TODO the value to use when the lab was not ordered: the investigators' fill-in (section 3.2), or the
    #      cohort median? Decide per lab. A lab left out of this dict is median-filled by impute_and_scale.
    #      (After the repair of step 2: a column that is still text cannot be filled with a number.)
    # "alb": ..., "pafi": ..., "bili": ..., "crea": ..., "bun": ..., "wblc": ..., "urine": ..., "ph": ..., "glucose": ...,
}


def add_knowledge(r):
    """r: a repaired frame (train or test). Returns a copy with the absences handled."""
    f = r.copy()
    # TODO 1, before any fill: the absences that carry information, one 0/1 column each.
    #      Pattern: f["x_missing"] = r["x"].isna().astype(float)
    #      Which absences? Fact 3, figures (g) and (h). One column per lab, or one per panel? Measure.
    # TODO 2: one ADL score from two reporters: the patient's answer when present, else the surrogate's (fact 3).
    #      Pattern: r["a"].where(r["a"].notna(), r["b"])
    # TODO 3: the labs that were not ordered, filled with NORMAL.
    for col, value in NORMAL.items():
        f[col] = f[col].fillna(value)
    return f


def build_features_3(train, test):
    R_train, R_test = repair(train), repair(test)
    F_train, F_test = add_knowledge(R_train), add_knowledge(R_test)
    cols = numeric_columns(F_train)
    return impute_and_scale(F_train[cols], F_test[cols])


X3_train, X3_test = build_features_3(train, test)
cv_auc(X3_train, y, "step 3: normal values, informative absences, one ADL score")
write_submission(X3_train, X3_test)

### Step 4: categories as categories, orders as orders

**What you know.** Fact 6: a diagnosis needs an encoding that gives each
group its own effect; a severity can keep its order as one number; encoding
a nominal column as an order asserts a direction that does not exist. The
vocabulary of each column is fixed on the train rows and applied to the
test rows, so a level absent from test still gets its column and a level
absent from train gets none. Figure (e) and the crosstab below it are the
evidence. Note what the SUPPORT model did with sex and race.

Two helpers are given: `one_hot` (one 0/1 column per level) and `ordinal`
(one column, the rank in an order you write). Which columns get which, and
which get neither, is the decision. `VOCAB` prints the levels of every text
column after your repair; if `sex` still shows eight levels, step 2 is not
finished.

**Aim for 0.916.**

In [ ]:
def one_hot(f, r, col, levels):
    """One 0/1 column per level of r[col]; `levels` is the vocabulary, fixed from the train rows."""
    for level in levels:
        f[f"{col}={level}"] = (r[col] == level).astype(float)


def ordinal(f, r, col, order):
    """One column: the position of r[col] in `order` (first = 0). Any other value -> NaN, median-filled later."""
    f[f"{col}_rank"] = r[col].map({level: i for i, level in enumerate(order)}).astype(float)


TEXT_COLUMNS = ["site", "sex", "race", "income", "dzgroup", "dzclass", "ca"]
VOCAB = {col: sorted(repair(train)[col].dropna().unique()) for col in TEXT_COLUMNS}
for col, levels in VOCAB.items():
    print(f"{col}: {levels}")


def add_categories(r, f):
    """r: the repaired frame; f: the feature frame so far. Returns f with the encoded columns added."""
    f = f.copy()
    # TODO for each text column decide: one_hot, ordinal, or leave it out. Say why in a comment (fact 6).
    #      one_hot(f, r, "<column>", VOCAB["<column>"])
    #      ordinal(f, r, "<column>", ["<lowest>", ..., "<highest>"])
    return f


def build_features_4(train, test):
    R_train, R_test = repair(train), repair(test)
    F_train, F_test = add_knowledge(R_train), add_knowledge(R_test)
    F_train, F_test = add_categories(R_train, F_train), add_categories(R_test, F_test)
    cols = numeric_columns(F_train)
    return impute_and_scale(F_train[cols], F_test[cols])


X4_train, X4_test = build_features_4(train, test)
cv_auc(X4_train, y, "step 4: categories encoded")
write_submission(X4_train, X4_test)

### Step 5: the bedside quantities

**What you know.** Fact 7 lists what a clinician computes from these
columns, with the published thresholds; section 3.5 repeats them. Fact 11:
`hday` is skewed. A straight line cannot compute a ratio, a threshold or a
logarithm from raw columns, so each formula is a candidate column with a
clinical reason behind it. Which ones pay, on this cohort, is what you
measure: add one family, run `cv_auc`, keep or drop.

The formulas assume °C and mg/dL (step 2) and complete values (step 3).
Guards: `meanbp` can be 0 and is real, so divide by `np.maximum(meanbp, 1)`
or cap the index; `wblc` can be 0, so use `np.log1p` or clip before a log.
`write_submission` refuses inf, and it is right to.

**Aim for 0.937.**

In [ ]:
def add_bedside(f):
    """f: the feature frame so far (repaired, filled). Returns f with the computed columns added."""
    f = f.copy()
    # TODO SIRS count, 0 to 4: temp > 38 or < 36; hrt > 90; resp > 20; wblc > 12 or < 4.
    # TODO modified shock index hrt / meanbp; the part above the threshold: np.maximum(0, msi - 1.3).
    # TODO BUN / creatinine ratio (both mg/dL); the part above 20.
    # TODO Berlin class from pafi: > 300 -> 0, 200 to 300 -> 1, 100 to 200 -> 2, < 100 -> 3.
    # TODO oliguria: urine < 500.
    # TODO hypoalbuminaemia np.maximum(0, 3.5 - alb); acidaemia np.maximum(0, 7.35 - ph); dysnatraemia |sod - 140|.
    # TODO logs: bili, crea, bun; white cells are abnormal in both directions: |log(wblc / 9)|.
    # TODO hday: np.log1p.
    return f


def build_features_5(train, test):
    R_train, R_test = repair(train), repair(test)
    F_train, F_test = add_knowledge(R_train), add_knowledge(R_test)
    F_train, F_test = add_categories(R_train, F_train), add_categories(R_test, F_test)
    F_train, F_test = add_bedside(F_train), add_bedside(F_test)
    cols = numeric_columns(F_train)
    return impute_and_scale(F_train[cols], F_test[cols])


X5_train, X5_test = build_features_5(train, test)
cv_auc(X5_train, y, "step 5: bedside formulas, logs, hinges")
write_submission(X5_train, X5_test)

### Step 6: scaling and selection

**What you know.** Fact 9: two copies of one signal do not give a penalised
line more to work with; the penalty splits one coefficient between them and
the fit gets less stable. Fact 8: scaling is a decision about the solver,
not about medicine, and `impute_and_scale` standardises every column,
indicators included. Fact 12: the professionals' model used constructed
variables, not raw columns, and not all of them.

The cell below lists the pairs of columns in your step-5 matrix that
correlate above 0.8, and the crosstab of the two diagnosis columns. Which
two columns say the same thing? Then try removing one family of features at
a time and keep only what the CV number needs.

**Aim for the step-5 number or a little above, with fewer columns.**

In [ ]:
corr = X5_train.corr().abs()
pairs = [(a, b, round(float(corr.loc[a, b]), 3))
         for i, a in enumerate(corr.columns) for b in corr.columns[i + 1:] if corr.loc[a, b] > 0.8]
print("pairs correlated above 0.8:", sorted(pairs, key=lambda t: -t[2])[:15])
pd.crosstab(train["dzgroup"], train["dzclass"])

In [ ]:
DROP = [
    # TODO the columns that repeat another column's information (fact 9), by name as they appear in X5_train.columns
]


def build_features_6(train, test):
    X_train, X_test = build_features_5(train, test)
    keep = [c for c in X_train.columns if c not in DROP]
    # TODO rows: the scorer accepts any subset of at least 4,000 train ids. Is there a row you do not trust? (optional)
    return X_train[keep], X_test[keep]


X6_train, X6_test = build_features_6(train, test)
cv_auc(X6_train, y, "step 6: redundant columns dropped")
write_submission(X6_train, X6_test)

### Your ladder

One row per `cv_auc` call, in the order you ran them. Put the leaderboard
number next to each step in a markdown cell: after step 2 the two should
agree to about 0.01.

In [ ]:
pd.DataFrame(results)

---

## 10. Rules, and where to go from here

**Rules.** Features must be computed from the columns you were given.
Target statistics are allowed only out-of-fold within train; a target
encoding fitted on the rows it encodes is caught by the train/test gap. No
external data, and no lookup of the public SUPPORT file: the label is
modelled and is not in it, and joining any outside file is forbidden. No
model other than the scorer's smuggled in as a feature. Your notebook is
part of the grade: it shows that the features are yours and why you built
them.

**Where to go from here.** The model will never change, so the question for
every column is: *what numbers would let one straight line use this?*
`EXPERTISE.md` gives the clinical reasons behind the choices that matter;
`DICTIONARY.md` tells you what each column is, how often it is filled, and
what its values look like. Measure one change at a time with `cv_auc`, and
keep what moves it. Once the post-outcome column is gone, the local number
and the leaderboard agree to about 0.01, so most of the work happens here,
not on the platform. The bar is **0.905**; the documented steps reach
0.937.